In [19]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import torchvision.models as models
from Loading_Dataset import LiverDataset

In [20]:
#Data augmentation and normalization for training
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [21]:
#Load dataset
root_dir = "./Dataset" 

dataset = LiverDataset(root_dir=root_dir, transform=transform)
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [22]:
#Load pre-trained ResNet-18 model
model = models.resnet18(pretrained=True)

c:\Users\minhp\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\minhp\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [23]:
#Replace the final fully connected layer for binary classification
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)  # binary classification

In [24]:
#Train the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
#Takes 10 mins + to run
epochs = 1

for epoch in range(epochs):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    acc = 100* correct/total


print(f'Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(dataloader):.4f}, Accuracy: {acc:.2f}%')

In [ ]:
#Feature extraction for vision encoder

#Removes the final fully connected layer to get feature vectors, removing classification layer
feature_extractor = nn.Sequential(*list(model.children())[:-1])  # Remove the final fully connected layer
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

with torch.no_grad():
    #Maps each image with the feature vector extracted from the ResNet-18 model
    features = feature_extractor(images)
    features = features.view(features.size(0), -1)

In [ ]:
#Features
print("Extracted feature:", features[0])



print(features.shape)

Extracted feature: tensor([0.5174, 1.0980, 0.9914, 1.0864, 0.9486, 0.9935, 0.0802, 0.6690, 1.0912,
        0.9789, 1.0940, 0.5427, 0.3223, 0.6013, 1.0240, 0.7858, 1.1666, 0.8742,
        1.5878, 0.0156, 1.6684, 1.0698, 0.5865, 1.2080, 0.7594, 1.4247, 0.9166,
        0.5992, 0.5030, 1.9075, 1.0915, 0.5095, 0.4203, 1.0601, 0.9056, 1.2685,
        1.3669, 0.2120, 1.0889, 0.5379, 0.8230, 0.8500, 0.9753, 1.0491, 1.0001,
        0.6297, 1.1900, 1.6949, 1.0502, 0.9475, 0.7433, 0.8024, 1.9052, 1.1901,
        1.1006, 0.8136, 1.4081, 1.1307, 1.0889, 1.5398, 1.3050, 1.2599, 0.4363,
        1.4422, 0.8486, 0.8760, 0.8270, 1.6257, 1.4451, 1.3748, 1.3168, 0.9178,
        1.0524, 1.1562, 1.5231, 1.1888, 0.5261, 0.8236, 1.4416, 1.2051, 0.0716,
        1.1616, 1.1776, 1.9107, 1.6653, 1.1949, 0.9370, 0.1181, 1.4833, 0.9033,
        1.1120, 0.7064, 1.0218, 0.8080, 0.9344, 0.9259, 1.1367, 0.1618, 0.2560,
        0.5193, 1.3398, 1.1336, 1.7106, 1.1587, 1.1368, 0.6215, 0.7009, 1.0544,
        0.9828, 1.100

torch.Size([11, 512]) = 11 images and each image dimension of 512


tensor([0.5174, 1.0980, 0.9914, ...]) = Image encoded into numbers